# Moduł 5: Dependency Injection w FastAPI

**Ćwiczenie:** #5 - Dependency Injection

---

## Spis treści

1. [Wprowadzenie - Database Session](#1-wprowadzenie---database-session)
2. [TRUE DI vs FastAPI Features](#2-true-di-vs-fastapi-features)
3. [Nested Dependencies - Authentication Chain](#3-nested-dependencies---authentication-chain)
4. [Best Practices](#4-best-practices)
5. [Demo Live Coding](#5-demo-live-coding)
6. [Podsumowanie](#6-podsumowanie)

---

## 1. Wprowadzenie - Database Session

### Po co Dependency Injection?

**Problem:** Powtarzający się kod w każdym endpoincie

**Rozwiązanie:** Oddeleguj tworzenie obiektu do funkcji i wstrzykuj go do endpointu

### Przykład

**BEZ Dependency Injection:**
```python
from sqlalchemy.orm import Session
from database import engine

@app.get("/tasks")
def get_tasks():
    with Session(engine) as db:  # ❌ Duplikacja!
        tasks = db.query(Task).all()
        return tasks

@app.post("/tasks")
def create_task(task: TaskCreate):
    with Session(engine) as db:  # ❌ Duplikacja!
        db_task = Task(name=task.name)
        db.add(db_task)
        db.commit()
        return db_task

@app.delete("/tasks/{task_id}")
def delete_task(task_id: int):
    with Session(engine) as db:  # ❌ Duplikacja!
        task = db.query(Task).filter(Task.id == task_id).first()
        db.delete(task)
        db.commit()
        return {"message": "Deleted"}
```

**Problem:**
- `with Session(engine) as db:` powtarza się w **każdym** endpoincie
- Jeśli chcesz zmienić sposób tworzenia sesji (np. dodać pool connection, timeout), musisz zmienić w **wielu miejscach**
- Trudne testowanie (jak zamockować session?)

**Z Dependency Injection:**
```python
from fastapi import Depends
from sqlalchemy.orm import Session
from database import engine
from typing import Generator

# ═══════════════════════════════════════════════════════════
# Dependency: tworzenie sesji w JEDNYM miejscu
# ═══════════════════════════════════════════════════════════
def get_db() -> Generator[Session, None, None]:
    """Dependency która tworzy i zamyka sesję bazodanową"""
    with Session(engine) as session:
        yield session  # FastAPI automatycznie zamknie session po requescie


# ═══════════════════════════════════════════════════════════
# Endpointy: session jest WSTRZYKIWANA
# ═══════════════════════════════════════════════════════════
@app.get("/tasks")
def get_tasks(db: Session = Depends(get_db)):  # ✅ DI!
    tasks = db.query(Task).all()
    return tasks

@app.post("/tasks")
def create_task(task: TaskCreate, db: Session = Depends(get_db)):  # ✅ DI!
    db_task = Task(name=task.name)
    db.add(db_task)
    db.commit()
    return db_task

@app.delete("/tasks/{task_id}")
def delete_task(task_id: int, db: Session = Depends(get_db)):  # ✅ DI!
    task = db.query(Task).filter(Task.id == task_id).first()
    db.delete(task)
    db.commit()
    return {"message": "Deleted"}
```

**Co się zmieniło?**
1. **Funkcja `get_db()`** - tworzy session w JEDNYM miejscu
2. **`db: Session = Depends(get_db)`** - FastAPI automatycznie wywołuje `get_db()` i wstrzykuje wynik do endpointu
3. **Endpointy** - nie tworzą session sami, tylko dostają go z zewnątrz

**To jest PRAWDZIWE Dependency Injection!**
- Endpoint nie tworzy swojej zależności (session)
- Zależność jest dostarczana z zewnątrz (przez `get_db()`)
- "Oddelegowujemy tworzenie sesji gdzie indziej i ją wstrzykujemy do endpointu"

### Korzyści DI:

**1. DRY (Don't Repeat Yourself)**
- Logika tworzenia session w jednym miejscu
- Zmiana w `get_db()` = zmiana wszędzie automatycznie

**2. Testability**
```python
# W testach możesz podmienić prawdziwą bazę na in-memory
def get_test_db():
    test_engine = create_engine("sqlite:///:memory:")
    with Session(test_engine) as session:
        yield session

app.dependency_overrides[get_db] = get_test_db  # Override!
```

**3. Flexibility**
```python
# Łatwo zmienić implementację bez zmian w endpointach
def get_db():
    # Dodaj connection pooling
    # Dodaj timeout
    # Dodaj read replicas
    with Session(engine) as session:
        yield session
```

**4. Separation of Concerns**
- Endpoint zajmuje się logiką biznesową (CRUD)
- Dependency zajmuje się infrastrukturą (database connection)

**Ciekawostka**. Type hints dla generatora

Generator[YieldType, SendType, ReturnType]

1. YieldType = Session
- Co generator yieluje (oddaje)
- W naszym przypadku: obiekt Session

2. SendType = None
- Co można wysłać do generatora przez .send(wartość)
- None = nie wysyłamy niczego (nie używamy .send())

3. ReturnType = None
- Co generator zwraca na końcu (przez return)
- None = brak wartości zwracanej (tylko yield)

Dla FastAPI dependencies:

Zawsze używamy Generator[Session, None, None] bo:
- Yielujemy Session
- Nigdy nie używamy .send()
- Nigdy nie zwracamy wartości (FastAPI ignoruje return)

---

## 2. TRUE DI vs FastAPI Features

### FastAPI używa nazwy "Dependency Injection", ale nie wszystko to prawdziwe DI!

FastAPI ma dwa różne mechanizmy które oba nazywa "DI":

| Aspekt | TRUE DI | FastAPI Feature (Query Params) |
|--------|---------|--------------------------------|
| **Główny cel** | Dostarczenie obiektu z zewnątrz | Parsowanie query params |
| **O co chodzi?** | Endpoint nie tworzy zależności sam | DRY dla walidacji parametrów |
| **Przykład** | Session, User, Config | Pagination, Filtering, Sorting |
| **Czy to prawdziwe DI?** | ✅ **TAK** | ⚠️ **NIE** - to feature FastAPI/Pydantic |

### Przykład 1: TRUE DI (Database Session)

```python
from sqlalchemy.orm import Session

def get_db() -> Generator[Session, None, None]:
    """Dependency tworzy session"""
    with Session(engine) as session:
        yield session  # Endpoint dostaje gotowy obiekt

@app.get("/tasks")
def get_tasks(db: Session = Depends(get_db)):
    #           ^^^ Endpoint NIE tworzy session sam!
    #               Session jest dostarczony z zewnątrz (injection)
    tasks = db.query(Task).all()
    return tasks
```

**To jest TRUE DI bo:**
- Endpoint **nie wie** jak tworzyć Session
- Session jest **wstrzykiwany** z zewnątrz przez `get_db()`
- Można łatwo **zamockować** w testach (dependency_overrides)

### Przykład 2: FastAPI Feature (Query Params Parsing)

```python
from pydantic import BaseModel

class Pagination(BaseModel):
    page: int = 1
    size: int = 10

@app.get("/tasks")
def get_tasks(params: Pagination = Depends()):
    #             ^^^ FastAPI parsuje query params do Pydantic model
    #                 NIE jest to prawdziwe DI - to walidacja parametrów!
    skip = (params.page - 1) * params.size
    return tasks[skip:skip+params.size]
```

**To NIE jest TRUE DI bo:**
- FastAPI tylko **parsuje** query params z URL do Pydantic model
- Endpoint **nie dostaje** obiektu z zewnątrz - FastAPI go tworzy z request params
- To jest **feature FastAPI/Pydantic** do walidacji parametrów, nie wzorzec DI

### Czy `params: Pagination` różni się od `params: Pagination = Depends()`?

**Dla Pydantic models w GET requestach - często nie ma różnicy:**

```python
# Oba parsują query params identycznie:
@app.get("/tasks")
def get_tasks(params: Pagination):  # ← Implicit query params
    pass

@app.get("/tasks")
def get_tasks(params: Pagination = Depends()):  # ← Explicit query params
    pass
```

**Ale są subtelne różnice:**

| Sytuacja | `params: Model` | `params: Model = Depends()` |
|----------|-----------------|-----------------------------|
| GET request | Query params | Query params ✅ |
| POST request | **Body (JSON)** ⚠️ | Query params (rzadkie) |
| Funkcja zamiast Model | ❌ Nie działa | ✅ Wymaga Depends() |
| use_cache control | ❌ Brak | ✅ Depends(..., use_cache=False) |

**Best practice:** Używaj `= Depends()` gdy:
- Chcesz być explicit o query params (czytelność)
- Używasz funkcji (wymagane)
- Chcesz kontrolować cachowanie

### Przykłady TRUE DI w FastAPI:

**1. Database Session** ✅
```python
def get_db():
    with Session(engine) as session:
        yield session

# Endpoint dostaje gotowy obiekt z zewnątrz
@app.get("/tasks")
def get_tasks(db: Session = Depends(get_db)):
    pass
```

**2. Current User (Authentication)** ✅
```python
def get_current_user(token: str, db: Session):
    # Weryfikuj token, pobierz user z bazy
    user = db.query(User).filter(User.id == user_id_from_token).first()
    return user

# Endpoint dostaje gotowy User object z zewnątrz
@app.get("/me")
def get_me(current_user: User = Depends(get_current_user)):
    return current_user
```

**3. Configuration Object** ✅
```python
def get_settings():
    return Settings()  # Pydantic Settings

# Endpoint dostaje gotowy config z zewnątrz
@app.get("/config")
def get_config(settings: Settings = Depends(get_settings)):
    return {"debug": settings.DEBUG}
```

### Przykłady FastAPI Features (NIE TRUE DI):

**1. Pagination** ⚠️
```python
def pagination(page: int = 1, size: int = 10):
    return {"skip": (page-1)*size, "limit": size}

# To tylko parsuje query params i oblicza skip/limit
# Endpoint NIE dostaje obiektu z zewnątrz - FastAPI tworzy go z params
@app.get("/tasks")
def get_tasks(p: dict = Depends(pagination)):
    pass
```

**2. Filtering** ⚠️
```python
def filters(category: str = None, in_stock: bool = None):
    return {"category": category, "in_stock": in_stock}

# To tylko walidacja query params, nie prawdziwe DI
@app.get("/products")
def get_products(f: dict = Depends(filters)):
    pass
```

### Podsumowanie:

**TRUE DI:**
- ✅ Database Session (`db: Session = Depends(get_db)`)
- ✅ Current User (`user: User = Depends(get_current_user)`)
- ✅ Configuration (`settings: Settings = Depends(get_settings)`)

**FastAPI Features (walidacja parametrów):**
- ⚠️ Pagination, Filtering, Sorting
- ⚠️ Query params parsing z Pydantic models

Obie techniki są użyteczne! Ale nazywanie ich wszystkich "Dependency Injection" może być mylące.

---

## 3. Nested Dependencies - Authentication Chain

### Dependencies mogą wywoływać inne dependencies

Najbardziej praktyczny przykład: **Authentication Chain**

```python
from fastapi import Depends, HTTPException, Header
from sqlalchemy.orm import Session
from jose import jwt, JWTError

# ═══════════════════════════════════════════════════════════
# Dependency Level 1: Database Session
# ═══════════════════════════════════════════════════════════
def get_db() -> Generator[Session, None, None]:
    """Tworzy session bazodanową"""
    with Session(engine) as session:
        yield session


# ═══════════════════════════════════════════════════════════
# Dependency Level 2: Extract Token from Header
# ═══════════════════════════════════════════════════════════
def get_token(authorization: str = Header(...)):
    """
    Wyciąga JWT token z Authorization header
    
    Header format: "Bearer <token>"
    """
    if not authorization.startswith("Bearer "):
        raise HTTPException(401, "Invalid authorization header")
    
    token = authorization.replace("Bearer ", "")
    return token


# ═══════════════════════════════════════════════════════════
# Dependency Level 3: Verify Token
# ═══════════════════════════════════════════════════════════
def verify_token(token: str = Depends(get_token)):
    """
    Weryfikuje JWT token i zwraca payload
    
    Zależy od get_token (nested dependency)
    """
    try:
        payload = jwt.decode(token, SECRET_KEY, algorithms=[ALGORITHM])
        user_id: int = payload.get("sub")
        if user_id is None:
            raise HTTPException(401, "Invalid token payload")
        return user_id
    except JWTError:
        raise HTTPException(401, "Invalid token")


# ═══════════════════════════════════════════════════════════
# Dependency Level 4: Get Current User
# ═══════════════════════════════════════════════════════════
def get_current_user(
    user_id: int = Depends(verify_token),
    db: Session = Depends(get_db)
):
    """
    Pobiera aktualnie zalogowanego użytkownika z bazy
    
    Zależy od:
    - verify_token (który zależy od get_token)
    - get_db
    """
    user = db.query(User).filter(User.id == user_id).first()
    if not user:
        raise HTTPException(404, "User not found")
    return user


# ═══════════════════════════════════════════════════════════
# Dependency Level 5: Check Admin
# ═══════════════════════════════════════════════════════════
def get_current_admin(current_user: User = Depends(get_current_user)):
    """
    Sprawdza czy aktualny użytkownik to admin
    
    Zależy od całego łańcucha:
    get_token → verify_token → get_current_user → get_current_admin
    """
    if not current_user.is_admin:
        raise HTTPException(403, "Admin access required")
    return current_user


# ═══════════════════════════════════════════════════════════
# Endpointy używające dependencies
# ═══════════════════════════════════════════════════════════
@app.get("/me")
def get_my_profile(current_user: User = Depends(get_current_user)):
    """
    FastAPI automatycznie wywołuje cały łańcuch:
    1. get_db() → tworzy session
    2. get_token() → wyciąga token z header
    3. verify_token(token) → weryfikuje token
    4. get_current_user(user_id, db) → pobiera użytkownika
    5. Endpoint dostaje current_user
    """
    return current_user


@app.get("/tasks")
def get_my_tasks(
    current_user: User = Depends(get_current_user),
    db: Session = Depends(get_db)
):
    """Pobierz taski aktualnego użytkownika"""
    tasks = db.query(Task).filter(Task.user_id == current_user.id).all()
    return tasks


@app.delete("/admin/users/{user_id}")
def delete_user(
    user_id: int,
    current_admin: User = Depends(get_current_admin),
    db: Session = Depends(get_db)
):
    """
    Tylko admin może usuwać użytkowników
    
    FastAPI automatycznie:
    1. Sprawdza czy token jest valid
    2. Pobiera current_user
    3. Sprawdza czy current_user.is_admin
    4. Jeśli nie admin → 403 Forbidden
    5. Jeśli admin → endpoint się wykonuje
    """
    user = db.query(User).filter(User.id == user_id).first()
    if not user:
        raise HTTPException(404, "User not found")
    
    db.delete(user)
    db.commit()
    return {"message": f"User {user_id} deleted"}
```

### Diagram łańcucha dependencies:

```
Request: GET /me
Header: Authorization: Bearer <token>

FastAPI automatycznie wywołuje:

1. get_token(authorization)
   ↓ zwraca: "eyJhbGc..."

2. verify_token(token="eyJhbGc...")
   ↓ zwraca: user_id=123

3. get_db()
   ↓ zwraca: <Session object>

4. get_current_user(user_id=123, db=<Session>)
   ↓ zwraca: <User id=123 email="john@example.com">

5. get_my_profile(current_user=<User>)
   ↓ zwraca: {"id": 123, "email": "john@example.com"}
```

### Korzyści Nested Dependencies:

**1. Separation of Concerns**
- Każda dependency ma jedno zadanie (single responsibility)
- `get_token` - tylko wyciąga token
- `verify_token` - tylko weryfikuje
- `get_current_user` - tylko pobiera user z bazy

**2. Reusability**
- `get_current_user` używana w wielu endpointach
- `get_current_admin` używana tylko tam gdzie potrzeba uprawnień admin

**3. Testability**
```python
# W testach możesz zamockować każdy poziom osobno
def mock_get_current_user():
    return User(id=1, email="test@test.com", is_admin=True)

app.dependency_overrides[get_current_user] = mock_get_current_user
```

**4. Automatic Validation**
- Jeśli token invalid → 401 (verify_token rzuca exception)
- Jeśli user nie istnieje → 404 (get_current_user rzuca exception)
- Jeśli nie admin → 403 (get_current_admin rzuca exception)
- Endpoint się **nie wykona** jeśli którykolwiek krok się nie powiedzie!

---

## 4. Best Practices

### ✅ Dobre praktyki:

**1. Używaj DI dla obiektów z zewnętrznych systemów**
```python
# ✅ Database session
def get_db():
    with Session(engine) as session:
        yield session

# ✅ Redis connection
def get_redis():
    redis = Redis(host="localhost")
    yield redis
    redis.close()

# ✅ External API client
def get_api_client():
    return APIClient(api_key=settings.API_KEY)
```

**2. Używaj type hints**
```python
# ✅ Dobre - type hints
def get_db() -> Generator[Session, None, None]:
    with Session(engine) as session:
        yield session

# ❌ Złe - brak type hints
def get_db():
    with Session(engine) as session:
        yield session
```

**3. Dokumentuj dependencies**
```python
# ✅ Dobre - z docstringiem
def get_current_user(
    token: str = Depends(get_token),
    db: Session = Depends(get_db)
) -> User:
    """
    Pobiera aktualnie zalogowanego użytkownika
    
    Args:
        token: JWT token z Authorization header
        db: Database session
    
    Returns:
        User object
    
    Raises:
        HTTPException 401: Invalid token
        HTTPException 404: User not found
    """
    # ...
```

**4. Używaj yield dla cleanup**
```python
# ✅ Dobre - yield zamknie session automatycznie
def get_db():
    with Session(engine) as session:
        yield session  # FastAPI zamknie po requescie

# ❌ Złe - musisz pamiętać o zamknięciu
def get_db():
    session = Session(engine)
    return session  # Kto zamknie session?
```

**5. Nazywaj dependencies opisowo**
```python
# ✅ Dobre - jasna nazwa
def get_current_user(): pass
def get_current_admin(): pass
def get_db(): pass

# ❌ Złe - niejasne nazwy
def dep1(): pass
def check(): pass
def get(): pass
```

**6. Oddzielaj TRUE DI od query params parsing**
```python
# ✅ TRUE DI - dla obiektów z zewnętrznych systemów
db: Session = Depends(get_db)
user: User = Depends(get_current_user)
settings: Settings = Depends(get_settings)

# ⚠️ Query params - dla walidacji parametrów (nie nazywaj tego DI)
page: int = Query(1, ge=1)
size: int = Query(10, ge=1, le=100)
# Możesz użyć Pydantic model jeśli chcesz DRY, ale to nie jest DI
```

### ❌ Unikaj:

**1. Nie modyfikuj globalnego stanu w dependencies**
```python
# ❌ Złe - side effects
counter = 0

def increment_counter():
    global counter
    counter += 1  # Side effect!
    return counter

# ✅ Dobre - pure function (lub kontrolowany state w bazie)
def get_timestamp():
    return datetime.now()  # Bez modyfikacji globalnego state
```

**2. Nie twórz dependency dla jednorazowej logiki**
```python
# ❌ Złe - dependency używana tylko raz
def get_special_value():
    return 42

@app.get("/special")
def endpoint(val = Depends(get_special_value)):
    return val

# ✅ Dobre - po prostu użyj w endpoincie
@app.get("/special")
def endpoint():
    return 42
```

**3. Nie twórz zbyt skomplikowanych dependencies**
```python
# ❌ Złe - zbyt dużo logiki
def complex_dependency(a, b, c, d, e, f):
    # 50 linii logiki...
    pass

# ✅ Dobre - rozbij na mniejsze
def simple_dep_1(a, b): pass
def simple_dep_2(c, d): pass
def combined_dep(
    dep1 = Depends(simple_dep_1),
    dep2 = Depends(simple_dep_2)
): pass
```

---

## 6. Podsumowanie

### Kluczowe zagadnienia:

**1. TRUE Dependency Injection**
- Endpoint NIE tworzy zależności sam - dostaje ją z zewnątrz
- Przykłady: Database Session, Current User, Configuration
- "Oddelegowujemy tworzenie obiektu gdzie indziej i go wstrzykujemy do endpointu"

**2. FastAPI używa nazwy "DI", ale nie wszystko to prawdziwe DI**
- TRUE DI: `db: Session = Depends(get_db)` - endpoint dostaje gotowy obiekt
- FastAPI feature: `params: Pagination = Depends()` - parsowanie query params, nie DI

**3. Nested Dependencies**
- Dependencies mogą wywoływać inne dependencies
- Przykład: `get_token → verify_token → get_current_user → get_current_admin`
- FastAPI automatycznie wywołuje cały łańcuch

**4. Korzyści DI:**
- ✅ **DRY** - Logika w jednym miejscu
- ✅ **Testability** - Łatwe mockowanie (dependency_overrides)
- ✅ **Flexibility** - Zmiana implementacji bez zmian w endpointach
- ✅ **Separation of Concerns** - Endpoint = logika biznesowa, Dependency = infrastruktura

**5. Best Practices:**
- Używaj DI dla obiektów z zewnętrznych systemów (DB, Redis, API clients)
- Używaj `yield` dla cleanup (automatyczne zamykanie połączeń)
- Dokumentuj dependencies z docstringami
- Nazywaj dependencies opisowo (`get_current_user`, nie `dep1`)
- Oddzielaj TRUE DI od query params parsing

### Najważniejsze:

**Database Session to najlepszy przykład TRUE DI w FastAPI:**
- Przed: `with Session(engine) as db:` w każdym endpoincie (duplikacja)
- Po: `db: Session = Depends(get_db)` - session wstrzykiwana z zewnątrz (DI)

**To jest istota Dependency Injection!**

---